## 03- Transformer Classifier using Sentence transformer + Setfit
In this notebook, we will do a fully neural method (Transformer-based sentence embeddings and setfit training for low-resource settings)

In [1]:
!pip install sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 171.5/171.5 kB 4.6 MB/s eta 0:00:00
  Using cached nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (23.7 MB)
  Using cached nvidia_cuda_runtime_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (823 kB)
  Using cached nvidia_cuda_cupti_cu12-12.1.105-py3-none-manylinux1_x86_64.whl (14.1 MB)
  Using cached nvidia_cudnn_cu12-8.9.2.26-py3-none-manylinux1_x86_64.whl (731.7 MB)
  Using cached nvidia_cublas_cu12-12.1.3.1-py3-none-manylinux1_x86_64.whl (410.6 MB)
  Using cached nvidia_cufft_cu12-11.0.2.54-py3-none-manylinux1_x86_64.whl (121.6 MB)
  Using cached nvidia_curand_cu12-10.3.2.106-py3-none-manylinux1_x86_64.whl (56.5 MB)
  Using cached nvidia_cusolver_cu12-11.4.5.107-py3-none-manylinux1_x86_64.whl (124.2 MB)
  Using cached nvidia_cusparse_cu12-12.1.0.106-py3-none-manylinux1_x86_64.whl (196.0 MB)
  Using cached nvidia_nccl_cu12-2.19.3-py3-none-manylinux1_x86_64.whl (166.0 MB)
  Using cached nvidia_nvtx_cu12-12.1.105-py3-none-many

In [2]:
!pip install wandb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 52.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 27.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 277.3/277.3 kB 35.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 8.1 MB/s eta 0:00:00


In [3]:
!pip install setfit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.9/75.9 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 30.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 14.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 16.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 23.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 20.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 401.2/401.2 kB 44.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface-hub 0.20.3
    Uninstalling huggingface-hub-0.20.3:
      Successfully uninstalled huggingface-hub-0.20.3


In [4]:
import pandas as pd
import numpy as np
# from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, f1_score, classification_report
# from sklearn.linear_model import LogisticRegression
# from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score


In [5]:
# loading api_key from local env credential file

# for local machine
# import json
# import urllib.parse

# with open('./env/credentials.json') as f:
#     login = json.load(f)

# api_key = login['api_key']
# print(len(api_key))

In [6]:
# for colab
from google.colab import userdata
api_key = userdata.get('wandb_capstone_api')
len(api_key)

40

In [44]:
# log in to your own wandb account
import wandb
# wandb.login(key='your_own_key')
wandb.login(key=api_key)

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

In [7]:
# log in to your own HuggingFace account
from huggingface_hub import notebook_login
notebook_login()

In [8]:
# for local machine
# train_fname = "../../data/splits/train.csv"
# dev_fname = "../../data/splits/dev.csv"

# train_fname = "../../data/splits/train_50pc.csv"
# dev_fname = "../../data/splits/dev_50pc.csv"

# train_df = pd.read_csv(train_fname)
# dev_df = pd.read_csv(dev_fname)

# print("Train len =", len(train_df))
# print("Dev df =", len(dev_df))

In [8]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# for Colab

# attach google drive

# train_fname = "/content/drive/MyDrive/capstone/data/splits/train.csv"
# dev_fname = "/content/drive/MyDrive/capstone/data/splits/dev.csv"

# train_fname = "/content/drive/MyDrive/capstone/data/splits/train_50pc.csv"
# dev_fname = "/content/drive/MyDrive/capstone/data/splits/dev_50pc.csv"

train_fname = "/content/drive/MyDrive/capstone/data/splits/train_50pc.csv"

dev_fname = "/content/drive/MyDrive/capstone/data/splits/dev_50pc.csv"

train_df = pd.read_csv(train_fname)
dev_df = pd.read_csv(dev_fname)

print("Train len =", len(train_df))

print("Dev df =", len(dev_df))

Train len = 83
Dev df = 83


In [10]:
train_extra_fname = "/content/drive/MyDrive/capstone/data/splits/train_pitt.csv"
train_extra_df = pd.read_csv(train_extra_fname)
print("Train extra len =", len(train_extra_df))



Train extra len = 372


In [11]:
train_df.head()

,text,target
0,I think you see going on in that picture. We'l...,AD
1,I'm gonna give you a picture that has a lot of...,AD
2,Ah the sinks running over waters come on all f...,CN
3,"Well, my boys trying to get some cookies and t...",AD
4,I'd like you to do is just look at the picture...,CN


In [12]:
train_extra_df.head()

,text,target
0,little boy getting in the cookie jar .\nand th...,CN
1,well ‡ the sink's overflowing .\nand &-uh the ...,CN
2,"this picture ?\nwell, there's a kid on a stool...",AD
3,everything I see in action or everything I +.....,CN
4,there's a little boy or> there's a little gir...,AD


In [13]:
dev_df.head()

,text,target
0,"Everything you see going on in this picture, g...",CN
1,"Now you just look at the picture. Okay, this i...",CN
2,Tell me everything that you see going on in th...,AD
3,Girls and boys getting her one that stores sto...,AD
4,"All right, we have a few more tasks to go thro...",CN


### Pre-processing
We will lowercase the text for better efficiency

In [14]:
train_df["text_lower"] = train_df["text"].apply(lambda text: text.lower().strip())
dev_df["text_lower"] = dev_df["text"].apply(lambda text: text.lower().strip())

In [15]:
train_extra_df["text_lower"] = train_extra_df["text"].apply(lambda text: text.lower().strip())


In [16]:
# stack up pre-train and train df
stack=False

if stack:
  train_df = pd.concat([train_df, train_extra_df], ignore_index=True, axis=0)

# # test with Pitt train set only
# train_df = extra_train_df

print(len(train_df))
train_df.head()

83


,text,target,text_lower
0,I think you see going on in that picture. We'l...,AD,i think you see going on in that picture. we'l...
1,I'm gonna give you a picture that has a lot of...,AD,i'm gonna give you a picture that has a lot of...
2,Ah the sinks running over waters come on all f...,CN,ah the sinks running over waters come on all f...
3,"Well, my boys trying to get some cookies and t...",AD,"well, my boys trying to get some cookies and t..."
4,I'd like you to do is just look at the picture...,CN,i'd like you to do is just look at the picture...


### Apply Sentence Transformer

We can use Sentence Transformer to convert text to a fixed-length embedding vector

In [17]:
# !pip install einops

In [18]:
# vectorizer = SentenceTransformer("sentence-t5-large") # works best
# vectorizer = SentenceTransformer("hkunlp/instructor-large")
# vectorizer = SentenceTransformer("hkunlp/instructor-xl")
# vectorizer = SentenceTransformer("all-MiniLM-L6-v2")
# vectorizer = SentenceTransformer("all-mpnet-base-v2")
# vectorizer = SentenceTransformer("sentence-transformers/paraphrase-mpnet-base-v2")

In [19]:
# corpus = train_df["text_lower"]
# print("Creating text embeddings...")
# X_train = vectorizer.encode(corpus)
# # X_train = vectorizer.encode(corpus,
# #                             # prompt="Does this person have Alzheimer's Disease (yes/no)?"
# #                             prompt="Represent the Medical paragraph for Alzheimer's Disease classification:"
# #                            )
# print("Completed")

In [20]:
# print(X_train.shape)

### Text embeddings analysis

We can compare a few sentences from AD / CN groups to see if they are different enough

In [21]:
# import random
# from sentence_transformers import util

In [22]:
# i = random.randint(0, 20)
# j = random.randint(0, 20)
# print(f"i = {i}, j = {j}")
# s1 = train_df.query("target=='AD'").iloc[i]["text_lower"]
# s2 = train_df.query("target=='AD'").iloc[j]["text_lower"]

# s3 = train_df.query("target=='CN'").iloc[i]["text_lower"]
# s4 = train_df.query("target=='CN'").iloc[j]["text_lower"]

# print("S1 =", s1[:200])
# print("S2 =", s2[:200])

# print("S3 =", s3[:200])
# print("S4 =", s4[:200])

# v1 = vectorizer.encode(s1)
# v2 = vectorizer.encode(s2)

# v3 = vectorizer.encode(s3)
# v4 = vectorizer.encode(s4)

# print("Same group similarity:")
# print("v1 vs v2 =", util.cos_sim(v1, v2).item()) # should be high
# print("v3 vs v4 =", util.cos_sim(v3, v4).item()) # should be high
# print("Cross group similarity:")
# print("v1 vs v3 =", util.cos_sim(v1, v3).item()) # should be low
# print("v2 vs v4 =", util.cos_sim(v2, v4).item()) # should be low


### Train using SetFit

We can try using SetFit for low-resource traiing

In [23]:
from sentence_transformers.losses import CosineSimilarityLoss
from setfit import SetFitModel, SetFitTrainer, Trainer, TrainingArguments
from datasets import Dataset

In [24]:
# convert Pandas dataframe to Dataset
train_set = Dataset.from_pandas(train_df, split="train")
eval_set = Dataset.from_pandas(dev_df, split="validation")

# print("Train len =", dataset["train"])

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [25]:
# model = SetFitModel.from_pretrained("sentence-transformers/paraphrase-mpnet-base-v2")
# model = SetFitModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2",
#                                     labels=["AD", "CN"])
model = SetFitModel.from_pretrained("all-mpnet-base-v2")
# model = SetFitModel.from_pretrained("all-MiniLM-L12-v2")
# model = SetFitModel.from_pretrained("sentence-t5-large") # too big to run on T4 / L4


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

model_head.pkl not found on HuggingFace Hub, initialising classification head with random weights. You should TRAIN this model on a downstream task to use it for predictions and inference.


In [26]:

# trainer = SetFitTrainer(
#     model=model,
#     train_dataset=train_set,
#     eval_dataset=eval_set,
#     # column_mapping={"text":"text_lower", "label":"target"},
#     column_mapping={"text":"text", "target":"label"},
#     loss_class=CosineSimilarityLoss,
#     batch_size=16,
#     # batch_size=32,
#     num_iterations=20,
#     metric="recall",
#     # load_best_model_at_the_end = True,
#     num_epochs=3  # 86% for 5 epochs, "all-MiniLM-L12-v2"
# )




In [27]:
args = TrainingArguments(
    batch_size=16,
    num_epochs=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,

)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_set,
    eval_dataset=eval_set,
    metric="recall",
    column_mapping={"text":"text", "target":"label"},

)

Applying column mapping to the training dataset
Applying column mapping to the evaluation dataset


Map:   0%|          | 0/83 [00:00<?, ? examples/s]

In [28]:
# metrics = trainer.evaluate()
# print("Evaluation before training =", metrics)

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [29]:
trainer.train()

***** Running training *****
  Num unique pairs = 3532
  Batch size = 16
  Num epochs = 10
  Total optimization steps = 2210
/usr/local/lib/python3.10/dist-packages/notebook/utils.py:280: DeprecationWarning: distutils Version classes are deprecated. Use packaging.version instead.
  return LooseVersion(v) >= LooseVersion(check)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize
wandb: Paste an API key from your profile and hit enter, or press ctrl+c to quit:

 ··········


wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


...

  0%|          | 0/225 [00:00<?, ?it/s]

  0%|          | 0/225 [00:00<?, ?it/s]

  0%|          | 0/225 [00:00<?, ?it/s]

  0%|          | 0/225 [00:00<?, ?it/s]

Loading best SentenceTransformer model from step 221.


In [30]:
metrics = trainer.evaluate()

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
***** Running evaluation *****


In [31]:
print("Evaluation after training =", metrics) # best so far: all-MiniLM-L12-v2, 5 epochs = 86%
# print(metrics)

Evaluation after training = {'recall': 0.9444444444444444}


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


![image.png](attachment:dcb1abd0-8bd2-4a16-95f0-5f363284391d.png)

In [32]:
# trainer.push_to_hub("minhngca/capstone_my_setfit_model")
trainer.push_to_hub("minhngca/capstone_all-mpnet-base-v2")

model_head.pkl:   0%|          | 0.00/6.99k [00:00<?, ?B/s]

Upload 2 LFS files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

CommitInfo(commit_url='https://huggingface.co/minhngca/capstone_paraphrase-mpnet-base-v2/commit/3d6b2fabb3bd913b6514ff4bd8920f1d4e9a1a88', commit_message='Add SetFit model', commit_description='', oid='3d6b2fabb3bd913b6514ff4bd8920f1d4e9a1a88', pr_url=None, pr_revision=None, pr_num=None)

In [45]:
wandb.finish()

eval/embedding_loss,▁▄▄▄▆▇▇▇██
train/embedding_loss,█▇▄▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
train/global_step,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇▇███
train/learning_rate,▁▃▄▇███▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▄▄▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁
eval/embedding_loss,0.2914
train/embedding_loss,0.0
train/epoch,10.0
train/global_step,2210
train/learning_rate,0.0
train_runtime,4052.7399


### Train Neural Classifier

We can now train a neural logistic regression classifier using processed X_train

In [ ]:
# y_train = train_df["target"]

# print("Training neural classifier...")
# hidden_size = 500
# clf = MLPClassifier(solver="lbfgs",
#                     # alpha=1e-5,
#                     hidden_layer_sizes=(hidden_size, hidden_size, hidden_size),
#                     # activation="relu",
#                     shuffle=True,
#                     # learning_rate="constant",
#                     # learning_rate_init=0.0001,
#                     # max_iter=100,
#                     random_state=42
#                    )
# clf.fit(X_train, y_train)
# print("Completed")

In [ ]:
# # trying ensemble such as XGBoost
# # from sklearn.ensemble import GradientBoostingClassifier
# from sklearn.svm import LinearSVC
# from sklearn.pipeline import make_pipeline
# from sklearn.preprocessing import StandardScaler
# from sklearn.ensemble import BaggingClassifier
# from sklearn.neighbors import KNeighborsClassifier
# from sklearn.linear_model import LogisticRegressionCV
# y_train = train_df["target"]

# print("Training classifier...")
# # hidden_size = 500
# # clf = make_pipeline(StandardScaler(),
# #                     LinearSVC(dual="auto", random_state=42)
# #                    )
# # clf = KNeighborsClassifier(n_neighbors=10)
# # clf = BaggingClassifier(KNeighborsClassifier(),
# #                         max_samples=0.5, max_features=8
# #                        )

# clf = make_pipeline(#StandardScaler(),
#                     LogisticRegressionCV(cv=10, random_state=42)
#                    )

# clf.fit(X_train, y_train)
# print("Completed")

In [34]:
# get train score
clf = model
train_preds = clf.predict(train_df["text"])
len(train_preds)


83

In [35]:
y_train = train_df["target"]
print(classification_report(y_train, train_preds))

              precision    recall  f1-score   support

          AD       1.00      1.00      1.00        40
          CN       1.00      1.00      1.00        43

    accuracy                           1.00        83
   macro avg       1.00      1.00      1.00        83
weighted avg       1.00      1.00      1.00        83



/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [36]:
# getting score on train set
# train_score = clf.score(X_train, y_train)
# print(f"Train accuracy = {train_score*100:.2f} %")

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


### Get accuracy using cross validation

In [37]:
# print("Performing cross-validation...")
# scores = cross_val_score(clf, X_train, y_train, cv=5)
# print("Completed")
# print(scores)

In [38]:
# print("Average =", np.average(scores))

In [39]:
# get precision / recall / f1
# y_pred = clf.predict(X_train)
# print(classification_report(y_train, y_pred))

### Now we can evaluate on dev set

In [40]:
# X_dev = vectorizer.encode(dev_df["text_lower"])
# y_dev = dev_df["target"]
# print("X_dev shape =", X_dev.shape)
# print("y_dev len =", len(y_dev))
dev_preds = clf.predict(dev_df["text"])
len(dev_preds)

83

In [41]:
# dev_score = clf.score(X_dev, y_dev)
y_dev = dev_df["target"]
from sklearn.metrics import accuracy_score
dev_score = accuracy_score(y_dev, dev_preds)
print(f"Dev accuracy = {dev_score*100:.2f} %")

Dev accuracy = 80.72 %


/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [42]:
# y_pred = clf.predict(X_dev)
# print(classification_report(y_dev, y_pred))
print(classification_report(y_dev, dev_preds))

              precision    recall  f1-score   support

          AD       0.94      0.70      0.80        47
          CN       0.71      0.94      0.81        36

    accuracy                           0.81        83
   macro avg       0.83      0.82      0.81        83
weighted avg       0.84      0.81      0.81        83



/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)


In [43]:
# best score so far on 50/50 dev set:
# 86%, with "all-MiniLM-L12-v2" x 5 epochs, recal ?

/usr/local/lib/python3.10/dist-packages/ipykernel/ipkernel.py:283: DeprecationWarning: `should_run_async` will not call `transform_cell` automatically in the future. Please pass the result to `transformed_cell` argument and any exception that happen during thetransform in `preprocessing_exc_tuple` in IPython 7.17 and above.
  and should_run_async(code)
